# ASMR v2 — driver

Thin driver for the reworked pipeline. All logic lives in the `asmr_*.py` modules; the
original `ASMR.ipynb` and `models_mammals.py` are left untouched.

**What changed and why** (see the module docstrings for detail):

1. `asmr_data.py` — v1 mis-indexed both participants (`ID_n` order vs CSV column order)
   and trials (item-ID order vs presentation order), so the regret signal compared one
   person against another and quoted the wrong item names. A hard assertion now
   re-derives the estimates from the narrative text and fails loudly if either drifts.
2. `asmr_likelihood.py` — Centaur's NLL is a probability *mass*; a continuous density is
   not comparable to it. A granularity-mixture reporting layer puts the cognitive model on
   the same footing. This moved the GCM from 7.63 to 5.47 nats/trial against Centaur's
   4.06.
3. `asmr_codegen.py` — a broken generation used to become a `1e12` sentinel NLL and get
   reported as "the AIC exploded". Every candidate is now validated first.
4. `asmr_srm.py` — points are selected by within-participant rank on a format-residualised
   Delta, and each one carries its cue vector, the model's prediction and the true value.
5. `asmr_fit.py` — multiple restarts, unconstrained parameters, plain BFGS, raw scale.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
np.seterr(over="ignore", invalid="ignore")

from asmr_data import load_aligned
from asmr_evaluate import (evaluate_library, load_library, reference_rows,
                           regret_diagnostics, response_style_summary, winner_table)
from asmr_fit import fit_all, granularity_weights, predictions
from asmr_models_seed import REFERENCE_MODELS, SEED_MODELS
from asmr_pipeline import make_generator, run

## 1. Load and verify alignment

In [ ]:
DOMAIN = "Mammals"
N_JOBS = 8          # participants are fitted in parallel processes

ds = load_aligned(DOMAIN)
print(response_style_summary(ds).to_string(index=False))
print()
print(reference_rows(ds).to_string(index=False))

## 2. Sanity-check the reporting layer before spending any GPU time

If the seed models do not move well below the v1 figure of 7.63 nats/trial, something is
wrong with the granularity weights and there is no point running the loop.

In [ ]:
from asmr_codegen import validate_model_code

pis = granularity_weights(ds)
for name, src in {**REFERENCE_MODELS, **SEED_MODELS}.items():
    rep = validate_model_code(src, ds.cues, ds.ex_cues, ds.ex_crit)
    f   = fit_all(rep.model_fn, rep.num_parameters, ds, pis=pis, n_restarts=2,
                  seed=0, n_jobs=N_JOBS, source=rep.source)
    print(f"{name:>10}: {f.nats_per_trial:.3f} nats/trial   AIC={f.aic:9.1f}")

## 3. Dry run — inspect the prompt without loading the LLM

In [ ]:
res = run(ds, seed="GCM", n_iterations=1, generate=None, n_jobs=N_JOBS,
          n_restarts=3, cv_folds=3, cv_restarts=2)

import numpy as np
from asmr_pipeline import OUT_DIR
z = np.load(OUT_DIR / f"{res['tag']}_iter0.npz", allow_pickle=True)
print(str(z["prompt"])[:3000])

## 4. Load the reasoning model

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 32768
MAX_NEW_TOKENS = 8192

llm, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-32B-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(llm)
generate = make_generator(llm, tokenizer, max_new_tokens=MAX_NEW_TOKENS)

## 5. Run the chains

One chain per (seed model, run id). Every validated model along the way is appended to
`../Data/Model Outputs/asmr_v2/library.jsonl`, which is what section 6 evaluates.

In [ ]:
results = []
for seed in ("GCM", "CAM", "RulEx-J", "MAPP"):
    for run_id in range(5):
        results.append(run(
            ds, seed=seed, run_id=run_id, n_iterations=5,
            generate=generate, tokenizer=tokenizer,
            n_jobs=N_JOBS, n_restarts=5, cv_folds=5, cv_restarts=3,
            max_new_tokens=MAX_NEW_TOKENS, max_seq_length=MAX_SEQ_LENGTH,
            temperature=0.7, residualize_format=True,
        ))

## 6. Evaluate the library

Each participant is assigned the library member with the best held-out log-likelihood.
The counts table is directly comparable to Table 2 of the manuscript (Mammals: 33 GCM,
9 MAPP, 3 RulEx-J, 3 guessing).

In [ ]:
library = load_library(domain=DOMAIN)
print(f"{len(library)} models in the library")

scores = evaluate_library(ds, library, n_folds=5, n_restarts=3, n_jobs=N_JOBS)
best, counts = winner_table(scores)
print(counts.to_string(index=False))

## 7. Did the regret signal stay honest?

In [ ]:
from asmr_codegen import validate_model_code
from asmr_srm import regret

rep = validate_model_code(res["best_source"], ds.cues, ds.ex_cues, ds.ex_crit)
fit = fit_all(rep.model_fn, rep.num_parameters, ds, pis=pis, n_restarts=3,
              seed=0, n_jobs=N_JOBS, source=rep.source)
preds = {p: predictions(rep.model_fn, fit, ds, p) for p in range(ds.n_subs)}
print(regret_diagnostics(regret(fit, ds), ds, preds).to_string(index=False))